# smp_jetmass_run2 — interactive run / parameter tuning

Tune the run parameters with the widgets below, **lock them** (writes
`configs/last_run.json`), then run any of the three channels (zjet / dijet / trijet)
through the shared `notebook_utils.run_from_config` — the same code path the CLI uses
(`python scripts/run_analysis_cli.py --config configs/last_run.json`).

In [ ]:
%load_ext autoreload
%autoreload 2

import json, sys, importlib
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display

repo_root = Path.cwd()
if not (repo_root / "smp_jetmass_run2").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import smp_jetmass_run2.notebook_utils as nbutils
importlib.reload(nbutils)

CONFIG_FILE = repo_root / "configs" / "last_run.json"
DEFAULTS = dict(nbutils.ANALYSIS_CONFIG_DEFAULTS)

# channel -> allowed datasets / modes
DATASETS = {
    "zjet":   ["pythia", "data", "herwig", "powheg", "st", "backgrounds", "pythia_local", "pythia2"],
    "dijet":  ["pythia", "mg_pythia8", "herwig", "data"],
    "trijet": ["pythia", "mg_pythia8", "herwig", "data"],
}
MODES = {
    "zjet":   ["minimal", "minimal_rho", "validation", "full", "mass_jk", "rho_jk", "reweight_pythia"],
    "dijet":  sorted(nbutils.HADRONIC_MODES),
    "trijet": sorted(nbutils.HADRONIC_MODES),
}
print("repo_root:", repo_root)

In [ ]:
# -------------------- widgets --------------------
def _load():
    cfg = DEFAULTS.copy()
    if CONFIG_FILE.exists():
        try: cfg.update(json.loads(CONFIG_FILE.read_text()))
        except Exception as e: print("Could not read", CONFIG_FILE, e)
    return cfg
cfg0 = _load()

channel_w   = widgets.Dropdown(options=nbutils.CHANNEL_OPTIONS, value=cfg0["channel"], description="channel")
dataset_w   = widgets.Dropdown(options=DATASETS[cfg0["channel"]], value=cfg0["dataset"] if cfg0["dataset"] in DATASETS[cfg0["channel"]] else DATASETS[cfg0["channel"]][0], description="dataset")
mode_w      = widgets.Dropdown(options=MODES[cfg0["channel"]], value=cfg0["mode"] if cfg0["mode"] in MODES[cfg0["channel"]] else MODES[cfg0["channel"]][0], description="mode")
era_w       = widgets.Dropdown(options=["2016APV","2016","2017","2018","all"], value=cfg0["era"], description="era")
syst_w      = widgets.Dropdown(options=["all_syst","minimal_syst","no_syst"], value=cfg0["systematic_profile"], description="systs")
executor_w  = widgets.Dropdown(options=["iterative","futures","dask-local","dask-lpc","dask-casa"], value=cfg0["executor_mode"], description="executor")
casa_w      = widgets.Checkbox(value=cfg0["casa"], description="casa")
test_w      = widgets.Checkbox(value=cfg0["test"], description="test (1 file/chunk)")
useDefault_w= widgets.Checkbox(value=cfg0["useDefault"], description="useDefault client")
chunksize_w = widgets.IntText(value=cfg0["chunksize"], description="chunksize")
chunktest_w = widgets.IntText(value=cfg0["chunksize_test"], description="chunk(test)")
group_w     = widgets.Dropdown(options=["all_in_one","per_group"], value=cfg0["group_mode"], description="group_mode")
redir_w     = widgets.Dropdown(options=["casa","lpc","fnal","local"], value=cfg0.get("redirector","casa"), description="redirector")

def _on_channel(change):
    ch = change["new"]
    dataset_w.options = DATASETS[ch]
    if dataset_w.value not in DATASETS[ch]: dataset_w.value = DATASETS[ch][0]
    mode_w.options = MODES[ch]
    if mode_w.value not in MODES[ch]: mode_w.value = MODES[ch][0]
channel_w.observe(_on_channel, names="value")

display(widgets.VBox([
    widgets.HBox([channel_w, dataset_w, mode_w]),
    widgets.HBox([era_w, syst_w, executor_w]),
    widgets.HBox([casa_w, test_w, useDefault_w]),
    widgets.HBox([chunksize_w, chunktest_w, group_w, redir_w]),
]))

In [ ]:
# -------------------- lock parameters --------------------
def get_config():
    redirector = redir_w.value
    cfg = {
        "channel": channel_w.value, "dataset": dataset_w.value, "mode": mode_w.value,
        "era": era_w.value, "systematic_profile": syst_w.value,
        "executor_mode": executor_w.value, "casa": casa_w.value, "test": test_w.value,
        "useDefault": useDefault_w.value, "chunksize": int(chunksize_w.value),
        "chunksize_test": int(chunktest_w.value), "group_mode": group_w.value,
        "redirector": redirector,
        "prependstr": nbutils.resolve_redirector_prepend(redirector),
    }
    return nbutils.validate_analysis_config(cfg)

cfg = get_config()
CONFIG_FILE.parent.mkdir(parents=True, exist_ok=True)
CONFIG_FILE.write_text(json.dumps(cfg, indent=2))
print("Locked config ->", CONFIG_FILE)
print(json.dumps(cfg, indent=2))

In [ ]:
# -------------------- run --------------------
# Uses the same path as the CLI. run_from_config creates+closes its own client
# from the config (executor_mode / casa). For a quick check set test=True above.
outputs, out = nbutils.run_from_config(cfg, repo_root=repo_root)
print("outputs:", outputs)

In [ ]:
# -------------------- inspect --------------------
if out is not None:
    for k, v in out.items():
        if hasattr(v, "axes"):
            print(f"{k}: axes={[a.name for a in v.axes]}")

In [ ]:
# example: project a reco hist (mass mode) and draw
import matplotlib.pyplot as plt
if out is not None and "ptjet_mjet_g_reco" in out:
    h = out["ptjet_mjet_g_reco"]
    h.project("mreco").plot()
    plt.title(f"{cfg['channel']} groomed jet mass (reco)")
    plt.show()